****Below is a practical table-level Delta property cheat sheet, including when to use it and important caveats.

1. Core Delta properties
Property	What it does	When to use
delta.autoOptimize.optimizeWrite	Optimizes file sizes during writes	High-frequency writes / streaming / many small files

delta.autoOptimize.autoCompact	Automatically compacts small files after writes	Streaming or incremental workloads

delta.enableChangeDataFeed	Tracks row-level changes	CDC, incremental ETL, audit, downstream processing

delta.enableDeletionVectors	Records row-level deletes/updates without rewriting entire files	Frequent DELETE, UPDATE, MERGE

delta.enableRowTracking	Maintains row identities across changes	Advanced incremental processing / row-level lineage use cases

delta.columnMapping.mode	Enables stable column IDs/names for schema evolution	Rename/drop columns, special characters, advanced schema evolution

delta.appendOnly	Prevents updates/deletes to the table	Immutable/event/log tables

delta.dataSkippingNumIndexedCols	Controls columns indexed for data skipping	Very wide tables where default data-skipping behavior needs tuning

delta.logRetentionDuration	Controls retention of transaction logs	Long historical metadata requirements

delta.deletedFileRetentionDuration	Controls how long old data files are retained before VACUUM can remove them	Time travel/recovery requirements

delta.checkpointInterval	Controls checkpoint frequency	Advanced tuning of very high transaction-volume tables

2. Auto Optimize properties
Optimized Write
ALTER TABLE sales
SET TBLPROPERTIES (
  'delta.autoOptimize.optimizeWrite' = 'true'
);

Think:

Before writing
      ↓
Optimize file sizes
      ↓
Write Delta files

Useful when:

Streaming
Frequent MERGE
Many small micro-batches
Auto Compaction
ALTER TABLE sales
SET TBLPROPERTIES (
  'delta.autoOptimize.autoCompact' = 'true'
);

Think:

Small files created
       ↓
Auto Compaction
       ↓
Larger files
Easy interview distinction

Optimized Write tries to prevent small files during the write; Auto Compaction combines small files after they are written.

3. Change Data Feed

This is one of the most important properties.

ALTER TABLE customer
SET TBLPROPERTIES (
  'delta.enableChangeDataFeed' = 'true'
);

It allows downstream processes to identify:

INSERT
UPDATE
DELETE

instead of repeatedly processing the entire table.

Example:

Customer Delta Table

ID   Name       City
1    Amit       Gwalior
2    Rahul      Delhi

Suppose:

UPDATE customer
SET City = 'Bhopal'
WHERE ID = 1;

With CDF enabled, downstream processing can consume the change rather than scanning the entire table.

When use?

Use it for:

CDC pipelines
Incremental ETL
Audit/change tracking
Feeding downstream systems
SCD processing
Important

CDF is not the same as Delta time travel.

Time Travel → "What did the table look like at that time?"

CDF → "What rows changed between versions/times?"
4. Deletion Vectors
ALTER TABLE sales
SET TBLPROPERTIES (
  'delta.enableDeletionVectors' = 'true'
);

Traditional deletion can require rewriting files.

For example:

File
 ├── Row A
 ├── Row B  ← DELETE
 ├── Row C
 └── Row D

Deletion vectors can record:

Row B → deleted

without immediately rewriting the entire Parquet file.

Useful for
DELETE
UPDATE
MERGE

especially when tables are large and modifications are frequent.

Important distinction

Deletion Vector:

How Delta represents row-level deletion/update efficiently.

CDF:

How downstream consumers discover row-level changes.

They solve different problems.

5. Row Tracking
ALTER TABLE customer
SET TBLPROPERTIES (
  'delta.enableRowTracking' = 'true'
);

Row tracking gives Delta rows stable identities that can be useful when determining which logical rows changed across table versions.

This is an advanced feature, so don't enable it just because "tracking is good."

Use it when your workload specifically benefits from row-level identity/change tracking and the relevant Databricks features require it.

6. Column Mapping

One of the most useful schema-evolution properties:

ALTER TABLE customer
SET TBLPROPERTIES (
  'delta.columnMapping.mode' = 'name'
);

It allows Delta to maintain stable column identity independently from the physical Parquet column representation.

Useful when you need operations such as:

Rename column
Drop column
Special characters in column names
More advanced schema evolution

For example:

Old:

customer_name

↓

New:

customer_full_name

Column mapping helps Delta understand that the logical column identity has changed without treating the physical Parquet representation as a completely unrelated column.

7. Append-only
ALTER TABLE event_log
SET TBLPROPERTIES (
  'delta.appendOnly' = 'true'
);

This means the table is intended to receive:

INSERT
INSERT
INSERT
INSERT

rather than:

UPDATE
DELETE
MERGE
Good use case

An immutable event table:

Kafka
  ↓
Events
  ↓
Delta

where historical events should never be modified.

Don't use it when

You need:

UPDATE
DELETE
MERGE
8. Data skipping

Delta maintains statistics that can help skip irrelevant files.

You may encounter:

delta.dataSkippingNumIndexedCols

This is relevant when dealing with very wide tables and tuning which columns have statistics available for skipping.

Conceptually:

Query:

WHERE customer_id = 100

             ↓

File statistics
             ↓
Can this file contain customer 100?
             ↓
YES → READ
NO  → SKIP

Don't confuse this with:

CLUSTER BY
Z-ORDER

Those organize data to make skipping more effective; statistics are what allow the engine to make pruning decisions.

9. Log retention
delta.logRetentionDuration

Example:

ALTER TABLE sales
SET TBLPROPERTIES (
  'delta.logRetentionDuration' = '30 days'
);

Delta has transaction logs:

_delta_log/
   000000.json
   000001.json
   000002.json
   ...

These contain the table's transaction history/metadata.

Longer retention can support historical operations and metadata recovery requirements.

But:

Longer log retention does not automatically mean old Parquet data files remain available.

That's where the next property matters.

10. Deleted file retention
delta.deletedFileRetentionDuration

Example:

ALTER TABLE sales
SET TBLPROPERTIES (
  'delta.deletedFileRetentionDuration' = '7 days'
);

This controls how long obsolete data files are retained before they become eligible for removal by:

VACUUM

Think:

UPDATE / DELETE
      ↓
Old file becomes obsolete
      ↓
Retained for configured period
      ↓
VACUUM
      ↓
Physical removal
Very important

VACUUM does not simply delete _delta_log files.

VACUUM primarily removes old data files that are no longer referenced by the current table state and are past the retention threshold.

11. Time Travel relationship

This is a common interview trap.

Suppose:

Version 100
Version 101
Version 102

You want:

SELECT *
FROM sales VERSION AS OF 100;

Time travel requires the necessary historical metadata and underlying data files to still exist.

Therefore:

Time Travel
     ↓
Transaction Log + Data Files
     ↓
VACUUM can eventually remove old files
     ↓
Old versions may no longer be queryable

So don't say:

"Increasing log retention alone guarantees time travel."

It doesn't.

12. Checkpoint interval
delta.checkpointInterval

Delta periodically creates checkpoint files so that readers don't have to replay every JSON transaction log from the beginning.

Conceptually:

000001.json
000002.json
...
000010.json
       ↓
checkpoint
       ↓
000011.json
000012.json
...

A checkpoint summarizes table state.

This is generally an advanced tuning property. You normally don't change it unless you have a specific transaction-log workload/problem.

13. A practical table design

Suppose your project has:

Auto Loader
     ↓
Bronze
     ↓
Silver
     ↓
Gold
Bronze

Usually:

Append-heavy
Schema evolution
Streaming
Raw data preservation

Potential considerations:

Optimized writes
Auto compaction
CDF

depending on downstream requirements.

Silver

Suppose you're doing:

MERGE
UPDATE
DELETE
SCD
Data cleansing

Potentially useful:

Deletion Vectors
CDF
Optimized Writes
Auto Compaction

depending on workload.

Gold

Suppose:

BI queries
Large analytical tables
Frequent filtering

Think about:

CLUSTER BY
Optimized Writes
Auto Compaction

and query-driven physical layout.

14. The properties you should NOT blindly enable

This is important in real projects.

Don't do:

SET TBLPROPERTIES (
 'delta.enableChangeDataFeed' = 'true',
 'delta.enableDeletionVectors' = 'true',
 'delta.enableRowTracking' = 'true',
 'delta.autoOptimize.optimizeWrite' = 'true',
 'delta.autoOptimize.autoCompact' = 'true'
);

just because all of them sound useful.

Instead ask:

What is my workload?
       ↓
Streaming?
MERGE?
CDC?
BI?
Append-only?
Schema evolution?
       ↓
Which property solves my actual problem?
15. The complete mental map

For your Databricks interviews, remember this:

                    DELTA TABLE
                         │
        ┌────────────────┼────────────────┐
        │                │                │
        ↓                ↓                ↓
   WRITE / FILE       CHANGES          SCHEMA
   MANAGEMENT         TRACKING         EVOLUTION
        │                │                │
        ├─ Optimized     ├─ CDF           └─ Column Mapping
        │  Write         │
        └─ Auto          ├─ Deletion
           Compaction   │  Vectors
                        │
                        └─ Row Tracking


        ┌─────────────────────────┐
        │       PERFORMANCE       │
        └─────────────────────────┘
                  │
                  ├─ Data Skipping
                  ├─ CLUSTER BY
                  └─ Z-ORDER


        ┌─────────────────────────┐
        │       LIFECYCLE         │
        └─────────────────────────┘
                  │
                  ├─ Log Retention
                  ├─ Deleted File Retention
                  ├─ VACUUM
                  └─ Checkpoints
⭐ Most important interview distinctions
Optimized Write
→ Better files while writing

Auto Compaction
→ Combines small files after writing

CLUSTER BY
→ Organizes data for locality/data skipping

Z-ORDER
→ Reorganizes existing data using multidimensional locality

CDF
→ Captures row-level changes for downstream consumers

Deletion Vector
→ Efficiently represents row-level deletes/updates

Column Mapping
→ Separates logical column identity from physical representation

VACUUM
→ Physically removes obsolete data files

Time Travel
→ Reads historical table versions when required log + data files exist****